# log-back — ex2: compose log_back and multiply_back through z = log(x*y)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `log-back`. Running the final beacon cell reports progress against the `Backprop: log_back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: log_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`log-back`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "log-back"
DD_SUBTOPIC = "Backprop: log_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## log_back composed with multiply_back — quick refresher

ex1 derived `log_back` from the local chain rule `d/dx log(x) = 1/x`. The deeper facet is **composition**: backward fns aren't just stand-alone — they get chained through the reverse pass when the forward op is composed.

Forward: `z = log(x * y)`. Two ops, three tensors. The reverse pass walks the graph end-first:

```
dL/dz   = 1                              # seed (ones_like)
dL/d(x*y) = log_back(dL/dz, z, x*y)      # = 1 / (x*y)
dL/dx   = multiply_back0(dL/d(x*y), x*y, x, y) = (1 / (x*y)) * y = 1/x
dL/dy   = multiply_back1(dL/d(x*y), x*y, x, y) = (1 / (x*y)) * x = 1/y
```

The closed-form gradients `dL/dx = 1/x` and `dL/dy = 1/y` are what torch.autograd produces — composition of the two back fns reproduces the global chain rule.

### Exercise 2 — compose log_back and multiply_back through z = log(x*y)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply log_back composed with multiply_back across a 2-op forward z = log(x*y) and show the result matches the closed-form gradients dL/dx = 1/x and dL/dy = 1/y.
> Keywords: log-back, chain-rule, composition, multiply-back
> ```

**KCs targeted:** `log-back`, `chain-rule-elementwise`

Implement `compose_log_multiply_back(x, y)` — returns `(dL_dx, dL_dy)` for the forward `z = log(x * y)` summed to scalar, computed by hand-composing two back fns.

The math (this is the LO):
```
z = log(x * y)
L = z.sum()              # so dL/dz = ones_like(z)
dL/d(x*y) = log_back(dL/dz, z, x*y)                   = ones / (x*y)
dL/dx     = multiply_back0(dL/d(x*y), x*y, x, y)      = (1/(x*y)) * y  = 1/x
dL/dy     = multiply_back1(dL/d(x*y), x*y, x, y)      = (1/(x*y)) * x  = 1/y
```

You implement THREE back fns + the composition:

1. `log_back(grad_out, out, x_arg)` — `grad_out / x_arg`.
2. `multiply_back0(grad_out, out, x_arg, y_arg)` — `grad_out * y_arg`.
3. `multiply_back1(grad_out, out, x_arg, y_arg)` — `grad_out * x_arg`.
4. `compose_log_multiply_back(x, y)`:
   - Compute `xy = x * y` and `z = log(xy)`.
   - Seed `dL_dz = t.ones_like(z)`.
   - Chain back through `log_back` and then `multiply_back{0,1}`.
   - Return `(dL_dx, dL_dy)` — both raw `torch.Tensor`.

Composition is what gives you the simple closed form: the algebra cancels even though no individual back fn 'knows' the final answer. Verify against `torch.autograd`.

Inputs strictly positive (so `log` is defined). Don't call `torch.autograd` in your implementation.

In [ ]:
def log_back(grad_out: Tensor, out: Tensor, x_arg: Tensor) -> Tensor:
    raise NotImplementedError()


def multiply_back0(grad_out: Tensor, out: Tensor, x_arg: Tensor, y_arg: Tensor) -> Tensor:
    raise NotImplementedError()


def multiply_back1(grad_out: Tensor, out: Tensor, x_arg: Tensor, y_arg: Tensor) -> Tensor:
    raise NotImplementedError()


def compose_log_multiply_back(x: Tensor, y: Tensor):
    """Hand-compose log_back ∘ multiply_back for z = log(x*y). Returns (dL/dx, dL/dy)."""
    raise NotImplementedError()


def _test_ex2():
    # --- back-fn correctness in isolation ---
    x = t.tensor([2.0, 4.0])
    y = t.tensor([3.0, 5.0])
    xy = x * y
    z = t.log(xy)
    g_log = log_back(t.ones(2), z, xy)
    assert t.allclose(g_log, 1.0 / xy, atol=1e-7), f'log_back wrong: {g_log}'
    g0 = multiply_back0(t.ones(2), xy, x, y)
    g1 = multiply_back1(t.ones(2), xy, x, y)
    assert t.allclose(g0, y), f'multiply_back0 wrong: {g0}'
    assert t.allclose(g1, x), f'multiply_back1 wrong: {g1}'

    # --- composition produces closed-form 1/x, 1/y ---
    x = t.tensor([1.0, 2.0, 4.0, 8.0])
    y = t.tensor([3.0, 5.0, 7.0, 11.0])
    dL_dx, dL_dy = compose_log_multiply_back(x, y)
    assert dL_dx.shape == x.shape, f'dL/dx shape: {dL_dx.shape}'
    assert dL_dy.shape == y.shape, f'dL/dy shape: {dL_dy.shape}'
    expected_dx = 1.0 / x
    expected_dy = 1.0 / y
    assert t.allclose(dL_dx, expected_dx, atol=1e-6), (
        f'dL/dx mismatch: got {dL_dx}, expected {expected_dx}'
    )
    assert t.allclose(dL_dy, expected_dy, atol=1e-6), (
        f'dL/dy mismatch: got {dL_dy}, expected {expected_dy}'
    )

    # --- agreement with torch.autograd on the full composed loss ---
    x_ref = x.clone().requires_grad_(True)
    y_ref = y.clone().requires_grad_(True)
    loss = t.log(x_ref * y_ref).sum()
    loss.backward()
    assert t.allclose(dL_dx, x_ref.grad, atol=1e-6), 'composition disagrees with autograd on x'
    assert t.allclose(dL_dy, y_ref.grad, atol=1e-6), 'composition disagrees with autograd on y'

    # --- different shape to catch shape-loss bugs ---
    x2 = t.tensor([[1.0, 2.0], [4.0, 8.0]])
    y2 = t.tensor([[3.0, 5.0], [7.0, 11.0]])
    dL_dx2, dL_dy2 = compose_log_multiply_back(x2, y2)
    assert dL_dx2.shape == (2, 2)
    assert t.allclose(dL_dx2, 1.0 / x2, atol=1e-6)
    assert t.allclose(dL_dy2, 1.0 / y2, atol=1e-6)
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def log_back(grad_out: Tensor, out: Tensor, x_arg: Tensor) -> Tensor:
    return grad_out / x_arg


def multiply_back0(grad_out: Tensor, out: Tensor, x_arg: Tensor, y_arg: Tensor) -> Tensor:
    return grad_out * y_arg


def multiply_back1(grad_out: Tensor, out: Tensor, x_arg: Tensor, y_arg: Tensor) -> Tensor:
    return grad_out * x_arg


def compose_log_multiply_back(x: Tensor, y: Tensor):
    xy = x * y
    z = t.log(xy)
    # seed.
    dL_dz = t.ones_like(z)
    # step 1: back through log.
    dL_dxy = log_back(dL_dz, z, xy)
    # step 2: back through multiply (two parents → two back fns).
    dL_dx = multiply_back0(dL_dxy, xy, x, y)
    dL_dy = multiply_back1(dL_dxy, xy, x, y)
    return dL_dx, dL_dy
```

**Why composition yields the simple closed form.** The intermediate `dL/d(x*y) = 1/(x*y)` looks ugly, but the next back fn multiplies by the OTHER factor — `y` in the case of `dL/dx` — and the algebra cancels: `(1/(x*y)) * y = 1/x`. This is the whole point of automatic differentiation: each back fn stays local and simple, but the chain of compositions reproduces the global derivative without ever materializing the Jacobian.

**Why log_back doesn't read `out`.** Same observation as ex1, but the composition makes it concrete: `log_back(dL_dz, z, xy)` ignores `z` entirely. The signature carries `out` for dispatcher uniformity; the actual gradient computation only needs the input.

**Why both multiply_backs run on the SAME intermediate.** `multiply` has two tensor parents, both contributing to the same output. The reverse pass dispatches both back fns with the SAME `grad_out` (here `dL_dxy`), routing each to its corresponding parent. This is the canonical pattern for any K-parent op.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()